In [ ]:
!pip install langchain langchain-google-genai langgraph scikit-learn --quiet

In [2]:
import os
import json
from datetime import datetime, timedelta
from typing import Optional, TypedDict, Literal

import numpy as np
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

In [3]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [4]:
MODEL = 'gemini-2.5-flash'

# ChatGoogleGenerativeAI reads GOOGLE_API_KEY from the environment automatically
llm = ChatGoogleGenerativeAI(model=MODEL)

In [5]:
class EmailClassification(BaseModel):
    urgency: Literal['Low', 'Medium', 'High'] = Field(
        description=(
            'High: financial errors, service outages, data loss, security breaches. '
            'Medium: bugs blocking functionality, persistent API failures, account access issues. '
            'Low: how-to questions, feature requests, general enquiries.'
        )
    )
    topic: Literal['Account', 'Billing', 'Bug', 'Feature Request', 'Technical Issue'] = Field(
        description='Primary topic category of the support email.'
    )
    reasoning: str = Field(
        description='One sentence explaining the urgency and topic classification.'
    )

In [6]:
classify_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a customer support triage specialist. '
     'Classify the incoming support email by urgency and topic.'),
    ('human', '{email_text}'),
])
classify_chain = classify_prompt | llm.with_structured_output(EmailClassification)

draft_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a professional customer support agent. '
     'Write clear, concise, professional email responses.'),
    ('human',
     'Tone instruction: {tone_instruction}\n\n'
     'Customer email:\n{email_text}\n\n'
     'Knowledge base reference:\n{kb_context}\n\n'
     'Write only the email body. No subject line. No internal notes. '
     'No placeholder brackets such as [Name] or [Date].'),
])
draft_chain = draft_prompt | llm | StrOutputParser()

In [7]:
KNOWLEDGE_BASE = [
    {'id': 'kb001', 'topic': 'Account', 'title': 'Password Reset',
     'content': ('To reset your password, navigate to the login page and click Forgot Password. '
                 'Enter your registered email address. A reset link will arrive within 5 minutes. '
                 'Check your spam folder if you do not see it. The link expires after 1 hour.')},
    {'id': 'kb002', 'topic': 'Account', 'title': 'Update Account Email',
     'content': ('To update your account email, go to Settings > Account > Email. '
                 'Enter the new address and confirm with your password. '
                 'A verification email is sent. The change takes effect only after verification.')},
    {'id': 'kb003', 'topic': 'Billing', 'title': 'Duplicate or Incorrect Charges',
     'content': ('If you see duplicate or incorrect charges, our billing team will issue a full refund '
                 'within 3-5 business days. Provide your invoice number, charge date, and last four '
                 'digits of the payment method. Contact billing@company.com or open a billing ticket.')},
    {'id': 'kb004', 'topic': 'Billing', 'title': 'Cancel Subscription',
     'content': ('To cancel, navigate to Settings > Billing > Cancel Subscription. '
                 'Access continues until end of billing period. No partial refunds for unused time. '
                 'You may resubscribe at any time without data loss.')},
    {'id': 'kb005', 'topic': 'Bug', 'title': 'PDF Export Crash',
     'content': ('Known issue: PDF export may fail for documents over 50 MB or with unsupported fonts. '
                 'Workaround: split large documents before exporting. Fix targeted for next release. '
                 'Internal reference: BUG-2041.')},
    {'id': 'kb006', 'topic': 'Technical Issue', 'title': '504 Gateway Timeout on API Calls',
     'content': ('504 errors indicate the server did not respond in time. '
                 'Causes: large payloads, high server load, network instability. '
                 'Actions: (1) Exponential backoff with jitter. (2) Reduce payload size. '
                 '(3) Contact support with request IDs if errors persist.')},
    {'id': 'kb007', 'topic': 'Technical Issue', 'title': 'API Rate Limits',
     'content': ('Limits: Free (100 req/min), Pro (1000 req/min), Enterprise (custom). '
                 'Exceeding limits returns HTTP 429. '
                 'Best practices: cache responses, batch requests, retry with backoff. '
                 'Headers X-RateLimit-Remaining and X-RateLimit-Reset are in every response.')},
    {'id': 'kb008', 'topic': 'Feature Request', 'title': 'Submitting Feature Requests',
     'content': ('Feature requests are reviewed quarterly by the product team. '
                 'Submit via Help > Feedback or email product@company.com. '
                 'Upvote existing requests at roadmap.company.com.')},
    {'id': 'kb009', 'topic': 'Account', 'title': 'Two-Factor Authentication',
     'content': ('Enable 2FA under Settings > Security > Two-Factor Authentication. '
                 'Supported: Google Authenticator, Authy, and SMS. '
                 'Use backup codes if you lose your device. Contact support if codes are unavailable.')},
    {'id': 'kb010', 'topic': 'Billing', 'title': 'Subscription Plans and Upgrades',
     'content': ('Plans: Basic (29/month), Enterprise (custom). '
                 'Upgrades take effect immediately with prorated charges. '
                 'Downgrades apply at the next billing cycle. Annual billing saves 20%.')},
    {'id': 'kb011', 'topic': 'Technical Issue', 'title': 'Intermittent API Integration Failures',
     'content': ('Intermittent failures may result from connection pooling issues, DNS delays, '
                 'or upstream dependencies. '
                 'Steps: enable verbose logging, capture headers, check status.company.com. '
                 'Provide request IDs for server-side log correlation.')},
]

In [8]:
_kb_texts = [d['title'] + ' ' + d['content'] for d in KNOWLEDGE_BASE]
_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
_kb_matrix = _vectorizer.fit_transform(_kb_texts)

def search_knowledge_base(query: str, top_k: int = 3) -> list:
    q = _vectorizer.transform([query])
    scores = cosine_similarity(q, _kb_matrix)[0]
    idx = np.argsort(scores)[::-1][:top_k]
    return [{'id': KNOWLEDGE_BASE[i]['id'], 'topic': KNOWLEDGE_BASE[i]['topic'],
             'title': KNOWLEDGE_BASE[i]['title'], 'content': KNOWLEDGE_BASE[i]['content'],
             'score': round(float(scores[i]), 4)}
            for i in idx if scores[i] >= 0.05]

_t = search_knowledge_base('forgot password login')

In [9]:
_CONF = 0.15

def decide_escalation(classification: dict, kb_results: list) -> dict:
    u, t = classification['urgency'], classification['topic']
    s = kb_results[0]['score'] if kb_results else 0.0
    if u == 'High':
        return {'action': 'escalate', 'reason': f'High-urgency {t} — immediate human intervention required.'}
    if t == 'Bug' and (not kb_results or s < _CONF):
        return {'action': 'escalate', 'reason': 'Bug with no documented resolution. Routing to engineering triage.'}
    if t == 'Bug':
        return {'action': 'auto-reply', 'reason': f'Known bug with documented workaround (KB score: {s}).'}
    if t == 'Technical Issue' and s < _CONF:
        return {'action': 'escalate', 'reason': 'Complex technical issue, insufficient KB coverage. Routing to technical support.'}
    if t == 'Feature Request':
        return {'action': 'auto-reply', 'reason': 'Feature request — standard roadmap response.'}
    if u in ('Low', 'Medium') and s >= _CONF:
        return {'action': 'auto-reply', 'reason': f'KB answer available (score: {s}). Auto-reply appropriate.'}
    return {'action': 'escalate', 'reason': 'Insufficient KB coverage. Routing to human agent.'}

In [10]:
_FU = {
    ('escalate',   'Billing'):         (24, 'Follow up in 24 h to confirm refund or resolution.'),
    ('escalate',   'Bug'):             (48, 'Follow up in 48 h to report engineering fix status.'),
    ('escalate',   'Technical Issue'): (48, 'Follow up in 48 h with human agent update.'),
    ('escalate',   'Account'):         (24, 'Follow up in 24 h to confirm account access restored.'),
    ('auto-reply', 'Bug'):             (72, 'Follow up in 72 h to confirm workaround resolved issue.'),
    ('auto-reply', 'Technical Issue'): (72, 'Follow up in 72 h to confirm technical resolution.'),
    ('auto-reply', 'Account'):         (72, 'Follow up in 72 h if customer has not confirmed resolution.'),
    ('auto-reply', 'Billing'):         (48, 'Follow up in 48 h to confirm billing acknowledgement.'),
}

def schedule_followup(classification: dict, decision: dict) -> dict:
    key = (decision['action'], classification['topic'])
    if key in _FU:
        h, r = _FU[key]
        return {'scheduled': True,
                'followup_at': (datetime.now() + timedelta(hours=h)).strftime('%Y-%m-%d %H:%M'),
                'hours_from_now': h, 'reason': r}
    return {'scheduled': False, 'followup_at': None, 'hours_from_now': None,
            'reason': 'No follow-up required for this case type.'}

In [11]:
class SupportAgentState(TypedDict):
    email_id: str; email_from: str; email_subject: str; email_body: str
    classification: Optional[dict]; kb_results: Optional[list]
    decision: Optional[dict]; response_draft: Optional[str]
    followup: Optional[dict]; processed_at: Optional[str]

def _kb_ctx(kb): return ('\n\n'.join(f"[{r['title']}]\n{r['content']}" for r in kb)
                          if kb else 'No relevant KB articles found.')

def _tone(decision):
    if decision['action'] == 'escalate':
        return ('Acknowledge concern empathetically, apologise, inform the customer a specialist '
                'will follow up shortly. Do not attempt to resolve the issue yourself.')
    return ('Provide a direct resolution using KB content. Be concise and professional. '
            'No filler phrases.')

def classify_node(state: SupportAgentState) -> dict:
    result = classify_chain.invoke({'email_text': state['email_body']})
    return {'classification': result.model_dump()}

def search_kb_node(state: SupportAgentState) -> dict:
    return {'kb_results': search_knowledge_base(state['email_body'], top_k=3)}

def escalation_node(state: SupportAgentState) -> dict:
    return {'decision': decide_escalation(state['classification'], state['kb_results'])}

def draft_node(state: SupportAgentState) -> dict:
    return {'response_draft': draft_chain.invoke({
        'email_text': state['email_body'],
        'kb_context': _kb_ctx(state['kb_results']),
        'tone_instruction': _tone(state['decision']),
    })}

def followup_node(state: SupportAgentState) -> dict:
    return {'followup': schedule_followup(state['classification'], state['decision']),
            'processed_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

In [12]:
workflow = StateGraph(SupportAgentState)
for name, fn in [('classify', classify_node), ('search_kb', search_kb_node),
                  ('escalation', escalation_node), ('draft', draft_node), ('followup', followup_node)]:
    workflow.add_node(name, fn)

workflow.set_entry_point('classify')
workflow.add_edge('classify', 'search_kb')
workflow.add_edge('search_kb', 'escalation')
workflow.add_edge('escalation', 'draft')
workflow.add_edge('draft', 'followup')
workflow.add_edge('followup', END)

agent = workflow.compile()
print('LangGraph pipeline compiled.')

LangGraph pipeline compiled.


In [13]:
def process_email(email: dict) -> dict:
    init: SupportAgentState = {
        'email_id': email.get('id', 'N/A'), 'email_from': email.get('from', ''),
        'email_subject': email.get('subject', ''), 'email_body': email['body'],
        'classification': None, 'kb_results': None, 'decision': None,
        'response_draft': None, 'followup': None, 'processed_at': None,
    }
    f = agent.invoke(init)
    return {'email_id': f['email_id'], 'from': f['email_from'], 'subject': f['email_subject'],
            'classification': f['classification'],
            'kb_matches': [{'id': r['id'], 'title': r['title'], 'score': r['score']}
                           for r in (f['kb_results'] or [])],
            'decision': f['decision'], 'response_draft': f['response_draft'],
            'followup': f['followup'], 'processed_at': f['processed_at']}

def display_result(r: dict) -> None:
    S, S2 = '=' * 72, '-' * 72
    print(S)
    print(f"  {r['email_id']}  |  {r['from']}\n  {r['subject']}")
    print(S)
    c = r['classification']
    print(f"  URGENCY   : {c['urgency']}  |  TOPIC: {c['topic']}")
    print(f"  REASONING : {c['reasoning']}")
    print()
    print('  KB MATCHES:', ', '.join(f"{m['title']} ({m['score']})" for m in r['kb_matches']) or 'none')
    print(f"  DECISION  : {r['decision']['action'].upper()} — {r['decision']['reason']}")
    print()
    print('  RESPONSE DRAFT:')
    print(S2)
    print('\n'.join(f'  {l}' for l in r['response_draft'].splitlines()))
    print(S2)
    fu = r['followup']
    if fu['scheduled']:
        print(f"  FOLLOW-UP : {fu['followup_at']} ({fu['hours_from_now']}h) — {fu['reason']}")
    else:
        print(f"  FOLLOW-UP : None — {fu['reason']}")
    print(S)

In [14]:
SAMPLE_EMAILS = [
    {'id': 'EMAIL-001', 'from': 'alice@example.com', 'subject': 'How do I reset my password?',
     'body': 'Hi, I forgot my password and cannot log in. I tried Forgot Password but am not sure what happens next. Can you walk me through it?'},
    {'id': 'EMAIL-002', 'from': 'bob@example.com', 'subject': 'Export feature crashes on PDF',
     'body': 'Every time I export as PDF the app crashes immediately. Happened three times today on the latest Chrome version. This is blocking my work.'},
    {'id': 'EMAIL-003', 'from': 'carol@example.com', 'subject': 'Charged twice for subscription!',
     'body': 'I see two charges of $29 on May 3rd. One account, one subscription. Unacceptable. I need a refund immediately. Invoice: INV-98123.'},
    {'id': 'EMAIL-004', 'from': 'dan@example.com', 'subject': 'Feature request: dark mode',
     'body': 'Been using your mobile app for six months. A dark mode option would make a big difference for late-night use. Is it on the roadmap?'},
    {'id': 'EMAIL-005', 'from': 'eva@techcorp.com', 'subject': 'API 504 errors — intermittent',
     'body': 'Intermittent 504 timeouts on your API for 72 hours, ~15% of requests. Retry logic in place but issue persists. 50k req/day, impacting production. IDs: req_a1b2c3, req_d4e5f6.'},
]

In [ ]:
results = []
for email in SAMPLE_EMAILS:
    print(f"Processing {email['id']} ...")
    r = process_email(email)
    results.append(r)
    display_result(r)

In [ ]:
hdr = f"{'ID':<12} {'Urgency':<8} {'Topic':<22} {'Action':<12} {'Follow-up At'}"
print(hdr)
print('-' * len(hdr))
for r in results:
    fu = r['followup']['followup_at'] if r['followup']['scheduled'] else 'None'
    print(f"{r['email_id']:<12} {r['classification']['urgency']:<8} "
          f"{r['classification']['topic']:<22} {r['decision']['action']:<12} {fu}")